In [1]:
import json
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from datasets import Dataset

In [2]:
# Check CUDA / GPU

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("PyTorch CUDA:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.5.1+cu121
CUDA available: True
PyTorch CUDA: 12.1
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [3]:
DATA_PATH = Path(
    r"C:\dsarp_outputs\improved_dataset\improved_multimetric_arch_smell_dataset.jsonl"
)

TEXT_COLUMN = "input_text"
LABEL_COLUMN = "selected_refactoring_labels"

MODEL_OUTPUT_DIR = Path(
    r"C:\dsarp_outputs\models\distilbert_improved_strict_ranked_top5"
)

MODEL_NAME = "distilbert-base-uncased"

MAX_LENGTH = 256
RANDOM_STATE = 42

In [4]:
# Load dataset

df = pd.read_json(DATA_PATH, lines=True)

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

df = df.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN])
df = df[df[TEXT_COLUMN].astype(str).str.strip() != ""]
df = df[df[LABEL_COLUMN].astype(str).str.strip() != ""]

print("Rows after cleaning:", len(df))

display(df[[TEXT_COLUMN, LABEL_COLUMN]].head())

Rows: 38
Columns: ['repository', 'commit', 'parent_commit', 'author_date', 'subject', 'all_refactoring_labels', 'refactoring_details_json', 'architecture_smell', 'primary_metric', 'primary_metric_before', 'primary_metric_after', 'primary_metric_delta', 'overall_improved', 'affected_elements', 'improved_metric_count', 'worsened_metric_count', 'unchanged_metric_count', 'positive_delta_sum', 'negative_delta_sum', 'improved_metric_names', 'worsened_metric_names', 'java_files_before', 'java_files_after', 'java_files_delta', 'packages_before', 'packages_after', 'packages_delta', 'package_edges_before', 'package_edges_after', 'package_edges_delta', 'cyclic_packages_before', 'cyclic_packages_after', 'cyclic_packages_delta', 'max_fan_in_before', 'max_fan_in_after', 'max_fan_in_delta', 'max_fan_out_before', 'max_fan_out_after', 'max_fan_out_delta', 'unstable_dependencies_before', 'unstable_dependencies_after', 'unstable_dependencies_delta', 'hub_like_packages_before', 'hub_like_packages_after', 

,input_text,selected_refactoring_labels
0,Architecture smell: Hub-like Dependency. Prima...,Rename Variable|Change Return Type|Inline Vari...
1,Architecture smell: Cyclic Dependency. Primary...,Remove Parameter|Rename Variable|Inline Variab...
2,Architecture smell: Hub-like Dependency. Prima...,Move Class|Rename Method|Change Method Access ...
3,Architecture smell: Cyclic Dependency. Primary...,Move Source Folder
4,Architecture smell: Cyclic Dependency. Primary...,Rename Variable|Change Attribute Type|Change P...


In [5]:
# Inspect smell and label distribution

if "architecture_smell" in df.columns:
    display(df["architecture_smell"].value_counts())

label_distribution = (
    df[LABEL_COLUMN]
    .astype(str)
    .str.split("|")
    .explode()
    .str.strip()
    .value_counts()
)

print("Unique labels:", len(label_distribution))
display(label_distribution.head(30))

architecture_smell
Cyclic Dependency      23
Hub-like Dependency    12
Unstable Dependency     3
Name: count, dtype: int64

Unique labels: 39


selected_refactoring_labels
Move Class                         16
Rename Variable                    14
Change Variable Type               14
Change Method Access Modifier      10
Change Parameter Type               8
Remove Parameter                    6
Move Method                         6
Change Return Type                  5
Rename Method                       5
Move And Rename Class               5
Add Parameter                       5
Extract Variable                    4
Inline Variable                     3
Change Thrown Exception Type        3
Change Attribute Type               3
Rename Parameter                    3
Parameterize Variable               3
Extract Method                      3
Replace Attribute With Variable     2
Move Source Folder                  2
Add Method Annotation               2
Move And Rename Method              2
Move Attribute                      2
Rename Package                      2
Inline Method                       2
Split Parameter       

In [6]:
# Create multi-label vocabulary

all_labels = sorted({
    label.strip()
    for labels in df[LABEL_COLUMN].astype(str)
    for label in labels.split("|")
    if label.strip()
})

label2id = {label: i for i, label in enumerate(all_labels)}
id2label = {i: label for label, i in label2id.items()}

num_labels = len(all_labels)

print("Number of labels:", num_labels)
print(all_labels)

Number of labels: 39
['Add Method Annotation', 'Add Method Modifier', 'Add Parameter', 'Add Thrown Exception Type', 'Add Variable Modifier', 'Change Attribute Type', 'Change Method Access Modifier', 'Change Parameter Type', 'Change Return Type', 'Change Thrown Exception Type', 'Change Variable Type', 'Extract And Move Method', 'Extract Method', 'Extract Variable', 'Inline Method', 'Inline Variable', 'Invert Condition', 'Localize Parameter', 'Move And Inline Method', 'Move And Rename Class', 'Move And Rename Method', 'Move Attribute', 'Move Class', 'Move Method', 'Move Package', 'Move Source Folder', 'Parameterize Variable', 'Remove Attribute Modifier', 'Remove Method Annotation', 'Remove Parameter', 'Remove Parameter Annotation', 'Remove Thrown Exception Type', 'Rename Method', 'Rename Package', 'Rename Parameter', 'Rename Variable', 'Replace Attribute With Variable', 'Split Package', 'Split Parameter']


In [7]:
# Convert labels to multi-hot vectors

def encode_labels(label_string):
    vector = np.zeros(num_labels, dtype=np.float32)

    for label in str(label_string).split("|"):
        label = label.strip()
        if label in label2id:
            vector[label2id[label]] = 1.0

    return vector.tolist()


df["labels"] = df[LABEL_COLUMN].apply(encode_labels)

display(df[[TEXT_COLUMN, LABEL_COLUMN, "labels"]].head())

,input_text,selected_refactoring_labels,labels
0,Architecture smell: Hub-like Dependency. Prima...,Rename Variable|Change Return Type|Inline Vari...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, ..."
1,Architecture smell: Cyclic Dependency. Primary...,Remove Parameter|Rename Variable|Inline Variab...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,Architecture smell: Hub-like Dependency. Prima...,Move Class|Rename Method|Change Method Access ...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, ..."
3,Architecture smell: Cyclic Dependency. Primary...,Move Source Folder,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,Architecture smell: Cyclic Dependency. Primary...,Rename Variable|Change Attribute Type|Change P...,"[0.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, ..."


In [8]:
# Train / validation / test split

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=RANDOM_STATE,
    shuffle=True,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=RANDOM_STATE,
    shuffle=True,
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 26
Validation: 6
Test: 6


In [9]:
# Convert to Hugging Face datasets

train_dataset = Dataset.from_pandas(train_df[[TEXT_COLUMN, "labels"]].rename(columns={TEXT_COLUMN: "text"}))
val_dataset = Dataset.from_pandas(val_df[[TEXT_COLUMN, "labels"]].rename(columns={TEXT_COLUMN: "text"}))
test_dataset = Dataset.from_pandas(test_df[[TEXT_COLUMN, "labels"]].rename(columns={TEXT_COLUMN: "text"}))

In [10]:
# Load tokenizer and model

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    problem_type="multi_label_classification",
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
# Tokenize text

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )


train_dataset = train_dataset.map(tokenize_batch, batched=True)
val_dataset = val_dataset.map(tokenize_batch, batched=True)
test_dataset = test_dataset.map(tokenize_batch, batched=True)

# Remove raw text column so Trainer only sees tensors it needs.
train_dataset = train_dataset.remove_columns(["text"])
val_dataset = val_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])

train_dataset.set_format("torch")
val_dataset.set_format("torch")
test_dataset.set_format("torch")

Map:   0%|          | 0/26 [00:00<?, ? examples/s]

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

In [12]:
# Metrics using default threshold 0.5

def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probabilities = sigmoid(logits)
    predictions = (probabilities >= 0.5).astype(int)

    return {
        "micro_f1": f1_score(labels, predictions, average="micro", zero_division=0),
        "macro_f1": f1_score(labels, predictions, average="macro", zero_division=0),
        "micro_precision": precision_score(labels, predictions, average="micro", zero_division=0),
        "micro_recall": recall_score(labels, predictions, average="micro", zero_division=0),
    }

In [13]:
# Training arguments

training_args = TrainingArguments(
    output_dir=str(MODEL_OUTPUT_DIR / "checkpoints"),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8 if torch.cuda.is_available() else 4,
    per_device_eval_batch_size=8 if torch.cuda.is_available() else 4,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,
    logging_steps=10,
)

In [14]:
# Train

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,Micro Precision,Micro Recall
1,No log,0.662892,0.135593,0.024420,0.121212,0.153846
2,No log,0.630358,0.100000,0.012821,0.142857,0.076923
3,0.663798,0.610467,0.137931,0.017094,0.666667,0.076923
4,0.663798,0.599882,0.137931,0.017094,0.666667,0.076923
5,0.605022,0.594870,0.137931,0.017094,0.666667,0.076923


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=20, training_loss=0.6344098091125489, metrics={'train_runtime': 87.73, 'train_samples_per_second': 1.482, 'train_steps_per_second': 0.228, 'total_flos': 8616062407680.0, 'train_loss': 0.6344098091125489, 'epoch': 5.0})

In [15]:
# Evaluate on test set using threshold 0.5

test_results = trainer.evaluate(test_dataset)
test_results

Training Loss,Validation Loss,Epoch,Micro F1,Macro F1,Micro Precision,Micro Recall
0.605022,0.610518,5,0.083333,0.008547,0.200000,0.052632


{'eval_loss': 0.6105179190635681,
 'eval_micro_f1': 0.08333333333333333,
 'eval_macro_f1': 0.008547008547008546,
 'eval_micro_precision': 0.2,
 'eval_micro_recall': 0.05263157894736842}

In [16]:
# Top-k evaluation

pred = trainer.predict(test_dataset)

logits = pred.predictions
true_labels = pred.label_ids
probs = sigmoid(logits)


def top_k_predictions(probs, k=5):
    preds = np.zeros_like(probs, dtype=int)
    top_indices = np.argsort(probs, axis=1)[:, -k:]

    for i, indices in enumerate(top_indices):
        preds[i, indices] = 1

    return preds


for k in [1, 2, 3, 5]:
    preds = top_k_predictions(probs, k=k)

    print("Top-k:", k)
    print("Micro F1:", f1_score(true_labels, preds, average="micro", zero_division=0))
    print("Macro F1:", f1_score(true_labels, preds, average="macro", zero_division=0))
    print("Precision:", precision_score(true_labels, preds, average="micro", zero_division=0))
    print("Recall:", recall_score(true_labels, preds, average="micro", zero_division=0))
    print()

Top-k: 1
Micro F1: 0.08
Macro F1: 0.007326007326007326
Precision: 0.16666666666666666
Recall: 0.05263157894736842

Top-k: 2
Micro F1: 0.12903225806451613
Macro F1: 0.014652014652014652
Precision: 0.16666666666666666
Recall: 0.10526315789473684

Top-k: 3
Micro F1: 0.10810810810810811
Macro F1: 0.014652014652014652
Precision: 0.1111111111111111
Recall: 0.10526315789473684

Top-k: 5
Micro F1: 0.12244897959183673
Macro F1: 0.020146520146520148
Precision: 0.1
Recall: 0.15789473684210525



In [17]:
# Threshold tuning

for threshold in [0.1, 0.2, 0.3, 0.4, 0.5]:
    preds = (probs >= threshold).astype(int)

    print("Threshold:", threshold)
    print("Micro F1:", f1_score(true_labels, preds, average="micro", zero_division=0))
    print("Macro F1:", f1_score(true_labels, preds, average="macro", zero_division=0))
    print("Precision:", precision_score(true_labels, preds, average="micro", zero_division=0))
    print("Recall:", recall_score(true_labels, preds, average="micro", zero_division=0))
    print()

Threshold: 0.1
Micro F1: 0.15019762845849802
Macro F1: 0.13003663003663002
Precision: 0.0811965811965812
Recall: 1.0

Threshold: 0.2
Micro F1: 0.15019762845849802
Macro F1: 0.13003663003663002
Precision: 0.0811965811965812
Recall: 1.0

Threshold: 0.3
Micro F1: 0.15019762845849802
Macro F1: 0.13003663003663002
Precision: 0.0811965811965812
Recall: 1.0

Threshold: 0.4
Micro F1: 0.15
Macro F1: 0.12576312576312576
Precision: 0.08144796380090498
Recall: 0.9473684210526315

Threshold: 0.5
Micro F1: 0.08333333333333333
Macro F1: 0.008547008547008546
Precision: 0.2
Recall: 0.05263157894736842



In [18]:
# Visual comparison on test set

def decode_true_labels(label_vector):
    return [
        id2label[i]
        for i, value in enumerate(label_vector)
        if value == 1
    ]


def decode_top_k(prob_vector, k=5):
    top_indices = prob_vector.argsort()[-k:][::-1]

    return [
        {
            "label": id2label[int(i)],
            "score": float(prob_vector[i]),
        }
        for i in top_indices
    ]


comparison_rows = []

test_df_reset = test_df.reset_index(drop=True)

for i in range(len(test_df_reset)):
    actual = decode_true_labels(true_labels[i])
    predicted = decode_top_k(probs[i], k=5)

    predicted_labels = [item["label"] for item in predicted]

    overlap = sorted(set(actual) & set(predicted_labels))
    missed = sorted(set(actual) - set(predicted_labels))
    extra = sorted(set(predicted_labels) - set(actual))

    comparison_rows.append({
        "input_text": test_df_reset.iloc[i][TEXT_COLUMN],
        "actual_labels": " | ".join(actual),
        "predicted_top5": " | ".join([
            f"{item['label']} ({item['score']:.3f})"
            for item in predicted
        ]),
        "matched": " | ".join(overlap),
        "missed_actual": " | ".join(missed),
        "extra_predicted": " | ".join(extra),
        "num_matched": len(overlap),
        "num_actual": len(actual),
    })

comparison_df = pd.DataFrame(comparison_rows)

pd.set_option("display.max_colwidth", 300)

display(comparison_df[[
    "input_text",
    "actual_labels",
    "predicted_top5",
    "matched",
    "missed_actual",
    "extra_predicted",
]].head(20))

,input_text,actual_labels,predicted_top5,matched,missed_actual,extra_predicted
0,"Architecture smell: Cyclic Dependency. Primary metric: cyclic_packages. Primary metric changed from 13 to 11, delta 2. Affected elements: org.apache.maven.artifact, org.apache.maven.artifact.handler.manager, org.apache.maven.artifact.manager, org.apache.maven.artifact.metadata, org.apache.maven....",Change Parameter Type | Extract Variable,Change Variable Type (0.507) | Rename Variable (0.490) | Split Parameter (0.488) | Split Package (0.487) | Remove Thrown Exception Type (0.485),,Change Parameter Type | Extract Variable,Change Variable Type | Remove Thrown Exception Type | Rename Variable | Split Package | Split Parameter
1,"Architecture smell: Cyclic Dependency. Primary metric: cyclic_packages. Primary metric changed from 20 to 19, delta 1. Affected elements: org.apache.maven, org.apache.maven.artifact, org.apache.maven.artifact.construction, org.apache.maven.artifact.factory, org.apache.maven.artifact.handler.mana...",Add Parameter | Change Attribute Type | Change Method Access Modifier | Move And Rename Class | Move Method,Change Variable Type (0.506) | Rename Variable (0.496) | Split Parameter (0.492) | Split Package (0.489) | Remove Thrown Exception Type (0.484),,Add Parameter | Change Attribute Type | Change Method Access Modifier | Move And Rename Class | Move Method,Change Variable Type | Remove Thrown Exception Type | Rename Variable | Split Package | Split Parameter
2,"Architecture smell: Cyclic Dependency. Primary metric: cyclic_packages. Primary metric changed from 20 to 16, delta 4. Affected elements: org.apache.ant.core.execution, org.apache.ant.core.types, org.apache.ant.engine, org.apache.ant.tasks, org.apache.myrmidon.components.builder, org.apache.myrm...",Move Class,Change Variable Type (0.513) | Rename Variable (0.492) | Add Method Annotation (0.489) | Change Method Access Modifier (0.488) | Split Parameter (0.484),,Move Class,Add Method Annotation | Change Method Access Modifier | Change Variable Type | Rename Variable | Split Parameter
3,"Architecture smell: Cyclic Dependency. Primary metric: cyclic_packages. Primary metric changed from 29 to 28, delta 1. Affected elements: org.apache.camel, org.apache.camel.bam, org.apache.camel.bam.model, org.apache.camel.bam.processor, org.apache.camel.bam.rules, org.apache.camel.builder, org....",Change Variable Type | Move Class | Move Package | Rename Method | Rename Variable,Change Variable Type (0.506) | Rename Variable (0.494) | Split Parameter (0.489) | Add Method Annotation (0.485) | Change Method Access Modifier (0.483),Change Variable Type | Rename Variable,Move Class | Move Package | Rename Method,Add Method Annotation | Change Method Access Modifier | Split Parameter
4,"Architecture smell: Cyclic Dependency. Primary metric: cyclic_packages. Primary metric changed from 50 to 48, delta 2. Affected elements: org.apache.camel, org.apache.camel.bam, org.apache.camel.bam.model, org.apache.camel.bam.processor, org.apache.camel.bam.rules, org.apache.camel.builder, org....",Remove Attribute Modifier,Change Variable Type (0.503) | Rename Variable (0.494) | Split Parameter (0.489) | Add Method Annotation (0.481) | Split Package (0.480),,Remove Attribute Modifier,Add Method Annotation | Change Variable Type | Rename Variable | Split Package | Split Parameter
5,"Architecture smell: Cyclic Dependency. Primary metric: cyclic_packages. Primary metric changed from 10 to 9, delta 1. Affected elements: org.apache.tika.config, org.apache.tika.detect, org.apache.tika.extractor, org.apache.tika.io, org.apache.tika.metadata, org.apache.tika.mime, org.apache.tika....",Add Parameter | Add Thrown Exception Type | Change Attribute Type | Change Parameter Type | Rename Variable,Change Variable Type (0.492) | Split Parameter (0.488) | Split Package (0.484) | Add Method Annotation (0.484) | Rename Variable (0.481),Rename Variable,Add Parameter | Add Thrown Exception Type | Change Attribute Type | Change 

In [19]:
# Save comparison table

MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

comparison_df.to_csv(
    MODEL_OUTPUT_DIR / "test_set_prediction_comparison.csv",
    index=False,
)

In [20]:
# Save final model and label mappings

FINAL_MODEL_DIR = MODEL_OUTPUT_DIR / "final_model"
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(FINAL_MODEL_DIR))
tokenizer.save_pretrained(str(FINAL_MODEL_DIR))

with open(FINAL_MODEL_DIR / "labels.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "label2id": label2id,
            "id2label": id2label,
            "label_column": LABEL_COLUMN,
            "text_column": TEXT_COLUMN,
            "model_name": MODEL_NAME,
            "max_length": MAX_LENGTH,
        },
        f,
        indent=2,
    )

print("Saved final model to:", FINAL_MODEL_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved final model to: C:\dsarp_outputs\models\distilbert_improved_strict_ranked_top5\final_model


In [21]:
# Load saved model and make suggestions, just creating suggestions on the first instance of the dataset and using the input_text, so one input smell description

from transformers import AutoTokenizer, AutoModelForSequenceClassification

loaded_tokenizer = AutoTokenizer.from_pretrained(str(FINAL_MODEL_DIR))
loaded_model = AutoModelForSequenceClassification.from_pretrained(str(FINAL_MODEL_DIR))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loaded_model.to(device)
loaded_model.eval()


def suggest_refactorings(input_text, top_k=5):
    encoded = loaded_tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
    )

    encoded = {key: value.to(device) for key, value in encoded.items()}

    with torch.no_grad():
        outputs = loaded_model(**encoded)
        probabilities = torch.sigmoid(outputs.logits[0]).cpu().numpy()

    top_indices = probabilities.argsort()[-top_k:][::-1]

    return [
        {
            "label": id2label[int(i)],
            "score": float(probabilities[i]),
        }
        for i in top_indices
    ]


sample_input = df.iloc[0][TEXT_COLUMN]
suggest_refactorings(sample_input, top_k=5)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'Add Method Annotation', 'score': 0.495750367641449},
 {'label': 'Split Parameter', 'score': 0.49068698287010193},
 {'label': 'Change Variable Type', 'score': 0.4862113296985626},
 {'label': 'Remove Thrown Exception Type', 'score': 0.4791242480278015},
 {'label': 'Split Package', 'score': 0.47552546858787537}]